In [26]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from pydantic import BaseModel,Field

load_dotenv()

llm=ChatOpenAI(model="deepseek-chat",
               base_url=os.getenv("DEEPSEEK_BASE_URL"),
               api_key=os.getenv("DEEPSEEK_API_KEY"),
               temperature=0)

In [27]:
from typing import TypedDict
from langchain_core.messages import SystemMessage,HumanMessage
from IPython.display import display,Image


def parent_node(state):
    messages=state['parent_input']
    result=llm.invoke(messages)
    return {'final_answer':result}

class ParentState(TypedDict):
    parent_input:str
    final_answer:str
    
class SubState(TypedDict):
    final_answer:str
    summary_answer:str

def subgraph_node1(state):
    system_prompt="将收到的消息总结，总结成不超过30个字"
    answer=state['final_answer']
    result=llm.invoke([SystemMessage(content=system_prompt),answer])
    print('subgraph_node1',result)
    return {'summary_answer':answer.content,'final_answer':result.content}

def subgraph_node2(state):
    system_prompt="分析拿到的回答和总结后的回答，进行打分，满分10分"
    result=llm.invoke([SystemMessage(content=system_prompt),HumanMessage(content=state['summary_answer']),
                HumanMessage(content=state['final_answer'])])
    return {'final_answer':result}

In [28]:
from langgraph.graph import END, START, StateGraph


subgraph_builder=StateGraph(SubState)

subgraph_builder.add_node('subgraph_node1',subgraph_node1)
subgraph_builder.add_node('subgraph_node2',subgraph_node2)

subgraph_builder.add_edge(START,'subgraph_node1')
subgraph_builder.add_edge('subgraph_node1','subgraph_node2')
subgraph_builder.add_edge('subgraph_node2',END)

subgraph=subgraph_builder.compile()

In [29]:
parent_builder=StateGraph(ParentState)

parent_builder.add_node('parent_node',parent_node)
parent_builder.add_node('subgraph',subgraph)

parent_builder.add_edge(START,'parent_node')
parent_builder.add_edge('parent_node','subgraph')
parent_builder.add_edge('subgraph',END)

parent_graph=parent_builder.compile()

In [35]:
# display(Image(parent_graph.get_graph(xray=True).draw_mermaid_png()))
print(parent_graph.get_graph().draw_mermaid())


# from langchain_core.runnables.graph import MermaidDrawMethod

# display(Image(
#     parent_graph.get_graph().draw_mermaid_png(
#         draw_method=MermaidDrawMethod.PYPPETEER
#     )
# ))


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parent_node(parent_node)
	subgraph(subgraph)
	__end__([<p>__end__</p>]):::last
	__start__ --> parent_node;
	parent_node --> subgraph;
	subgraph --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [31]:
async for chunk in parent_graph.astream({"parent_input":"你好，介绍一下你自己"},stream_mode="values"):
    print(chunk)

{'parent_input': '你好，介绍一下你自己'}
{'parent_input': '你好，介绍一下你自己', 'final_answer': AIMessage(content='你好呀！很高兴认识你！😊\n\n我是**DeepSeek**，由深度求索公司创造的AI助手。让我给你介绍一下我的“特长”：\n\n✨ **我的能力**：\n- **纯文本对话**：擅长回答各种问题，从日常闲聊到专业知识都可以\n- **文件处理**：支持上传图片、PDF、Word、Excel、PPT等文件，能从中提取文字信息帮你分析\n- **超长上下文**：拥有1M的上下文窗口，可以一次性处理像《三体》三部曲那么大体量的内容\n- **联网搜索**：需要时可以帮你搜索最新信息（需要你手动开启这个功能哦）\n\n🎯 **我的特点**：\n- **完全免费**：没错，不收费！尽管外面谣言四起，但我确实是免费的\n- **多平台使用**：网页版、App版都有，App还支持语音输入\n- **知识更新**：我的知识截止到2025年5月\n\n💡 **我不能做的**：\n- 不支持图像识别（虽然能读取图片中的文字，但看不懂图片内容本身）\n- 不能生成图片、视频等多媒体内容\n\n我的风格是热情、细腻，会尽力用你容易理解的方式回答问题。有什么想问我的吗？无论是学习、工作还是生活上的问题，我都很乐意帮忙！🌟', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 274, 'prompt_tokens': 8, 'total_tokens': 282, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26